In [0]:
from pyspark.sql.functions import *

from delta.tables import DeltaTable

In [0]:
base_adls2_path = "abfss://customer360unitycatalog@kmstorage9490.dfs.core.windows.net"

In [0]:
%sql
-- use catalog customer360_cata;

-- use schema bronze

In [0]:
source_df = spark.sql('select * from customer360_cata.bronze.locations')

In [0]:

# =====================================================
# CONFIG
# =====================================================

target_tbl = "customer360_cata.silver.locations"

# =====================================================
# SOURCE DATA
# =====================================================

source_df = source_df.withColumn(
    "hash_value",
    sha2(
        concat_ws(
            "|",
            col("city"),
            col("state"),
            col("country"),
            col("zipcode")
        ),
        256
    )
)

# =====================================================
# TABLE EXISTS
# =====================================================

if spark.catalog.tableExists(target_tbl):

    print("Table exists")

    delta_table = DeltaTable.forName(spark, target_tbl)

    # =================================================
    # CURRENT ACTIVE RECORDS
    # =================================================

    current_df = spark.table(target_tbl) \
        .filter(col("is_active") == True) \
        .withColumn(
        "hash_value",
        sha2(
            concat_ws(
                "|",
                col("city"),
                col("state"),
                col("country"),
                col("zipcode")
            ),
            256
        )
    )
    
    # =================================================
    # DETECT CHANGED RECORDS
    # =================================================

    changed_df = (
        source_df.alias("s")
        .join(
            current_df.alias("t"),
            "location_id"
        )
        .filter(
            col("s.hash_value") != col("t.hash_value")
        )
        .select("s.*")
    )
 
    # =================================================
    # BRAND NEW RECORDS
    # =================================================

    new_records_df = (
         source_df.alias("s")
        .join(
            current_df.alias("t"),
            "location_id",
            "leftanti"
        )
    )
  
    # =================================================
    # EXPIRE EXISTING RECORDS
    # =================================================

    expire_df = changed_df.select(
        col("location_id").alias("merge_key")
    )
    
    # =================================================
    # INSERT NEW + CHANGED RECORDS
    # =================================================

    insert_df = (
        new_records_df.unionByName(changed_df)
        .withColumn("effective_date", current_date())
        .withColumn("expiration_date", lit("9999-12-31"))
        .withColumn("is_active", lit(True))
        .withColumn("merge_key", lit(None))
    )
  
    # =================================================
    # STAGING DATAFRAME
    # =================================================

    staged_df = expire_df.unionByName(
        insert_df,
        allowMissingColumns=True
    )
    
    # =================================================
    # SINGLE MERGE (ENTERPRISE SCD2)
    # =================================================

    (
        delta_table.alias("t")
        .merge(
            staged_df.alias("s"),
            """
            t.location_id = s.merge_key
            AND t.is_active = true
            """
        )

        # Expire old record
        .whenMatchedUpdate(
            set={
                "is_active": lit(False),
                "expiration_date": current_date()
            }
        )

        # Insert new record
        .whenNotMatchedInsert(
            values={
                "location_id": "s.location_id",
                "city": "s.city",
                "state": "s.state",
                "country": "s.country",
                "zipcode": "s.zipcode",
                "effective_date": "s.effective_date",
                "expiration_date": "s.expiration_date",
                "is_active": "s.is_active"
            }
        )

        .execute()
    )

    print("SCD Type 2 Merge Completed")

else:

    print("Table does not exist")

    initial_df = (
        source_df.drop('hash_value')
        .withColumn("effective_date", current_date())
        .withColumn("expiration_date", lit("9999-12-31"))
        .withColumn("is_active", lit(True))
    )

    initial_df.write \
        .format("delta") \
        .mode("overwrite") \
            .option("overwriteSchema", "true")\
        .option("path", f"{base_adls2_path}/silver/locations") \
        .saveAsTable(target_tbl)

    print("Initial Load Completed")

In [0]:
%sql
-- select * from customer360_cata.silver.locations order by location_id 

In [0]:
%sql
 -- drop table if exists customer360_cata.silver.locations